# 03 · Train a U-Net baseline (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1ow1atency/floodfusion/blob/main/notebooks/03_train.ipynb)

**Only change the settings cell. Re-running after a disconnect resumes automatically. Never change hyperparameters based on test or bolivia results.**

Trains with `scripts/train.py` on a local copy of the data (fast disk), saves the run folder to Drive (`floodfusion/runs/<config name>_seed<SEED>`), then evaluates `best.pt` on valid, test and bolivia and plots the training curves. Needs a **T4 GPU** runtime and `00_download_data.ipynb` to have run once. The `GITHUB_TOKEN` (and optional `WANDB_API_KEY`) Colab secrets are described in the README, "Colab setup".

Settings: the only cell you should normally edit.

In [ ]:
# Options (configs/): s1_only, s2_only, early_fusion, fusion (FusionNet)
#   robustness training: s2_only_robust, early_fusion_robust, fusion_robust
#   ablation: fusion_concat_robust (fusion_robust with concat instead of cross-attention)
CONFIG = 'configs/s1_only.yaml'
SEED = 0                          # one run folder per config + seed
USE_WANDB = True                  # needs the WANDB_API_KEY Colab secret; metrics.csv is always written

GPU check.

In [ ]:
!nvidia-smi
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU available. In Colab: Runtime → Change runtime type → T4 GPU, then run all cells again.')
print('GPU:', torch.cuda.get_device_name(0))

Mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone the repo, or pull if it is already cloned. Uses the `GITHUB_TOKEN` Colab secret if set (see README "Colab setup"); otherwise tries a plain public clone.

In [ ]:
# Clone or update the repo in Colab. Shared cell: paste verbatim into every notebook
# (tests/test_notebooks.py checks the copies match notebooks/colab_clone_cell.py).
# Uses the Colab secret GITHUB_TOKEN if available (see README "Colab setup"). The token
# reaches git only through environment variables: never on the command line, never in
# .git/config (the remote URL stays token-free), and it is masked in any output.
import base64, os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/1ow1atency/floodfusion.git'
REPO_DIR = Path('/content/floodfusion')


def _github_token():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception as e:  # secret missing, notebook access off, or not in Colab
        print(f'GITHUB_TOKEN secret not available ({type(e).__name__}); trying a plain public clone.\n'
              'To add it: key icon in the left sidebar -> "Add new secret" -> name GITHUB_TOKEN, value = a\n'
              'GitHub token with read access to this repo -> switch "Notebook access" on -> re-run this cell.')
        return None
    return token.strip() or None


def _git(*args, token=None):
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}  # fail instead of waiting for a password
    secrets = []
    if token:
        b64 = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0=f'Authorization: Basic {b64}')
        secrets = [token, b64]
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode != 0:
        hint = '' if token else '\n(No token used. If the repo is not publicly visible, add the GITHUB_TOKEN secret.)'
        raise RuntimeError(f'git {args[0] if args[0] != "-C" else args[2]} failed:\n{out}{hint}')
    return out


def _clone_or_pull():
    version = tuple(int(x) for x in _git('--version').split()[2].split('.')[:2])
    if version < (2, 31):  # GIT_CONFIG_COUNT needs git >= 2.31
        raise RuntimeError(f'git >= 2.31 needed, found {version}')
    token = _github_token()
    if (REPO_DIR / '.git').exists():
        _git('-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL)  # keep the remote token-free
        print(_git('-C', str(REPO_DIR), 'pull', '--ff-only', token=token))
    else:
        print(_git('clone', REPO_URL, str(REPO_DIR), token=token))
    print('at', _git('-C', str(REPO_DIR), 'log', '--oneline', '-1'))


_clone_or_pull()

Install the package (base dependencies only, **not** the `clouds` extra: s2cloudless is never used in training).

In [ ]:
!pip install -q -e "{REPO_DIR}"
assert _exit_code == 0, 'pip install failed, see output above'

Copy the dataset from Drive to the local disk (much faster to read every epoch). Skipped if a complete copy is already there.

In [ ]:
import shutil, time

DRIVE_DATA = Path('/content/drive/MyDrive/floodfusion/data/sen1floods11')
LOCAL_DATA = Path('/content/data/sen1floods11')
RUNS_DIR = Path('/content/drive/MyDrive/floodfusion/runs')
FOLDERS, EXPECTED = ['S1Hand', 'S2Hand', 'LabelHand'], 446


def tif_counts(root):
    return {f: len(list((root / f).glob('*.tif'))) if (root / f).is_dir() else 0 for f in FOLDERS}


marker = LOCAL_DATA / '.copy_complete'
if marker.exists() and all(n == EXPECTED for n in tif_counts(LOCAL_DATA).values()):
    print(f'local copy already present: {LOCAL_DATA}')
else:
    t0 = time.time()
    for f in FOLDERS + ['splits']:
        shutil.copytree(DRIVE_DATA / f, LOCAL_DATA / f, dirs_exist_ok=True)
    print(f'copied {DRIVE_DATA} -> {LOCAL_DATA} in {time.time() - t0:.0f} s')
counts = tif_counts(LOCAL_DATA)
csvs = sorted(p.name for p in (LOCAL_DATA / 'splits').glob('*.csv'))
print('tif files per folder:', counts, '| split CSVs:', csvs)
assert all(n == EXPECTED for n in counts.values()), f'expected {EXPECTED} .tif files per folder, got {counts}'
assert len(csvs) == 4, f'expected 4 split CSVs, got {csvs}'
marker.touch()

Weights & Biases (optional). Without it, every metric is still saved to `metrics.csv` in the run folder.

In [ ]:
import os

WANDB_FLAG = '--no-wandb'
if USE_WANDB:
    try:
        from google.colab import userdata
        _key = (userdata.get('WANDB_API_KEY') or '').strip()
    except Exception as e:  # secret missing or notebook access off
        _key = ''
        print(f'WANDB_API_KEY secret not available ({type(e).__name__}).')
    if _key:
        import wandb
        os.environ['WANDB_API_KEY'] = _key  # inherited by the training process; never printed
        wandb.login(key=_key, relogin=True)
        WANDB_FLAG = ''
    del _key
if WANDB_FLAG:
    print('W&B is off: metrics are still saved to metrics.csv in the run folder (passing --no-wandb).')
else:
    print('W&B is on.')

Train. Output streams live, one line per epoch with its time. If this config + seed already has a `last.pt` on Drive, the run is resumed. `last.pt` is saved every `save_last_every` epochs (5 for the U-Net configs, 10 for the FusionNet configs) and on the final epoch, so a disconnect loses at most that many epochs.

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))  # the editable install is importable here only after a restart
from floodfusion.training import load_config, resolve_config, run_dir_for

cfg = resolve_config(load_config(REPO_DIR / CONFIG), out=str(RUNS_DIR), seed=SEED)
RUN_DIR = run_dir_for(cfg, smoke=False)
RESUME = '--resume' if (RUN_DIR / 'last.pt').exists() else ''
print(f'run folder: {RUN_DIR}')
print('found last.pt -> RESUMING this run (--resume)' if RESUME else 'no last.pt -> starting a new run')

!cd {REPO_DIR} && python scripts/train.py --config {CONFIG} --seed {SEED} --root "{LOCAL_DATA}" --out "{RUNS_DIR}" {WANDB_FLAG} {RESUME}
assert _exit_code == 0, 'train.py failed, see output above'

Evaluate `best.pt` (selected on valid only) on full chips of valid, test and bolivia. Results go to `eval_<split>/` in the run folder on Drive.

In [ ]:
for split in ['valid', 'test', 'bolivia']:
    !cd {REPO_DIR} && python scripts/evaluate.py --checkpoint "{RUN_DIR}/best.pt" --split {split} --root "{LOCAL_DATA}"
    assert _exit_code == 0, f'evaluate.py failed on {split}, see output above'

from floodfusion.evaluation import format_eval_table

print(format_eval_table(RUN_DIR))
print('\nTest and bolivia are for reporting only: never change hyperparameters based on them.')

Training curves from `metrics.csv` (also saved as `training_curves.png` in the run folder).

In [ ]:
from IPython.display import Image, display
from floodfusion.viz import plot_training_curves

curves = plot_training_curves(RUN_DIR / 'metrics.csv', RUN_DIR / 'training_curves.png', title=RUN_DIR.name)
display(Image(filename=str(curves)))

**After each run: empty Google Drive's Bin** (drive.google.com → Bin → Empty bin). `last.pt` is replaced every 5 epochs for the U-Net baselines (~294 MB) and every 10 for FusionNet (~590 MB); `best.pt` (~98 MB / ~196 MB) every time the valid IoU improves. On a Colab-mounted Drive the replaced versions can go to the Bin, where they still count against your 15 GB quota until the Bin is emptied (Drive deletes Bin items itself only after 30 days).